In [1]:
import torch, torch.onnx, tqdm, time, os, json, multiprocessing
import numpy as np
import pandas as pd
import pytorch_lightning as pl
from models.est_model import ResidualRegression, DNNRegression
from torch.utils.data import DataLoader, Dataset, random_split
from pytorch_lightning.callbacks import DeviceStatsMonitor, EarlyStopping
from sklearn.metrics import r2_score

In [2]:
monitor = DeviceStatsMonitor()

In [3]:
# set train setting
save_model = True
train_num_epoch = 50000
min_loss = 100
dl_workers = 0

In [4]:
gpu = torch.device('cuda')
torch.set_float32_matmul_precision('high')

In [5]:
# load hyperparameter of json file
with open('.' + os.sep + os.path.join('models', 'params_dnn_20220207-012403.json'), 'r') as file:
    hyper_params = json.load(file)

n_inputs = hyper_params['n_of_inputs']
n_outputs = hyper_params['n_of_outputs']
n_layers = hyper_params['n_of_hidden']

In [6]:
model = ResidualRegression(n_inputs, n_layers, n_outputs)
print(model)

ResidualRegression(
  (input_layer): Sequential(
    (0): Linear(in_features=4, out_features=88, bias=False)
  )
  (output_layer): Sequential(
    (0): Linear(in_features=88, out_features=22, bias=False)
  )
  (res_block_1): ResidualBlock(
    (active): ReLU()
    (layers): Sequential(
      (0): Linear(in_features=88, out_features=88, bias=True)
      (1): ReLU()
      (2): Linear(in_features=88, out_features=88, bias=True)
      (3): ReLU()
      (4): Linear(in_features=88, out_features=88, bias=True)
      (5): ReLU()
      (6): Linear(in_features=88, out_features=88, bias=True)
      (7): ReLU()
      (8): Linear(in_features=88, out_features=88, bias=True)
      (9): ReLU()
      (10): Linear(in_features=88, out_features=88, bias=True)
      (11): ReLU()
      (12): Linear(in_features=88, out_features=88, bias=True)
      (13): ReLU()
      (14): Linear(in_features=88, out_features=88, bias=True)
      (15): ReLU()
      (16): Linear(in_features=88, out_features=88, bias=True)
    

In [7]:
data = pd.read_csv('./resources/sim_data_edit.csv')
feature_names = ['lift_weight(ton)', 'lift_height(m)', 'rising_angle(deg)', 'swing_angle(deg)']
# target_names = ['left_ground_pressure_min(kg/cm2)', 'left_ground_pressure_max(kg/cm2)', 'left_pressure_length(m)', 'right_ground_pressure_min(kg/cm2)', 'right_ground_pressure_max(kg/cm2)', 'right_pressure_length(m)']
left_target_names = ['left-0.0m', 'left-0.675m', 'left-1.35m', 'left-2.025m', 'left-2.7m', 'left-3.375m', 'left-4.05m', 'left-4.725m', 'left-5.4m', 'left-6.075m', 'left-6.75m']
right_target_names = ['right-0.0m', 'right-0.675m', 'right-1.35m', 'right-2.025m', 'right-2.7m', 'right-3.375m', 'right-4.05m', 'right-4.725m', 'right-5.4m', 'right-6.075m', 'right-6.75m']
target_names = left_target_names + right_target_names

feature_data = []
target_data = []

for feature_name in feature_names:
    feature_data.append(data[feature_name])

for target_name in target_names:
    target_data.append(data[target_name])

feature_data = np.array(feature_data, dtype=np.float32).T
target_data = np.array(target_data, dtype=np.float32).T

class TensorDataset(Dataset):
    def __init__(self, feature, target):
        self.x_data = torch.tensor(feature)
        self.y_data = torch.tensor(target)
        self.len = self.x_data.shape[0]

    def __getitem__(self, index):
        return self.x_data[index], self.y_data[index]

    def __len__(self):
        return self.len

train_dataset = TensorDataset(feature_data, target_data)
train_data_loader = DataLoader(train_dataset, batch_size=len(train_dataset), num_workers=multiprocessing.cpu_count())

In [8]:
# train model
early_stop_callback = EarlyStopping(monitor='train_loss', mode='min', verbose=True, min_delta=0.001, patience=200)
trainer = pl.Trainer(callbacks=[early_stop_callback], accelerator='cuda', enable_progress_bar=True, max_epochs=10000)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [9]:
trainer.fit(model=model, train_dataloaders=train_data_loader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name         | Type          | Params
-----------------------------------------------
0 | input_layer  | Sequential    | 352   
1 | output_layer | Sequential    | 1.9 K 
2 | res_block_1  | ResidualBlock | 78.3 K
3 | res_block_2  | ResidualBlock | 78.3 K
4 | res_block_3  | ResidualBlock | 78.3 K
5 | loss         | MSELoss       | 0     
-----------------------------------------------
237 K     Trainable params
0         Non-trainable params
237 K     Total params
0.949     Total estimated model params size (MB)
/home/jinbeom/workspace/estimate_crawler_crane_ground_pressure/venv/lib/python3.8/site-packages/pytorch_lightning/loops/fit_loop.py:280: PossibleUserWarning: The number of training batches (1) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.
  rank_zero_warn(


Epoch 0: 100%|██████████| 1/1 [00:00<00:00,  1.60it/s, v_num=85]

Metric train_loss improved. New best score: 55.273


Epoch 2: 100%|██████████| 1/1 [00:00<00:00,  1.75it/s, v_num=85]

Metric train_loss improved by 31.302 >= min_delta = 0.001. New best score: 23.971


...

Epoch 688: 100%|██████████| 1/1 [00:00<00:00,  1.56it/s, v_num=85]

Metric train_loss improved by 0.001 >= min_delta = 0.001. New best score: 0.004


Epoch 888: 100%|██████████| 1/1 [00:00<00:00,  1.49it/s, v_num=85]

Monitored metric train_loss did not improve in the last 200 records. Best score: 0.004. Signaling Trainer to stop.


Epoch 888: 100%|██████████| 1/1 [00:00<00:00,  1.42it/s, v_num=85]


In [10]:
torch.save(model.state_dict(), './models/est_ground_pressure.pt')

In [11]:
model.eval()
for data in train_dataset:
    print(r2_score(data[1].detach().numpy() ,model(data[0]).detach().numpy()))
    print(model(data[0]).detach().numpy())
    print(data[1].detach().numpy())

0.9991152057403575
[ 4.0251832e+00  3.2941985e+00  2.5096419e+00  1.7910476e+00
  1.0204060e+00  3.3242819e-01  7.1557671e-02 -1.9333959e-03
 -5.1760197e-02 -5.5031300e-02  6.4524591e-02  3.9801724e+00
  3.2759509e+00  2.5301480e+00  1.8797197e+00  1.0874350e+00
  3.7642717e-01  9.9467039e-03 -4.1546330e-02 -4.4105142e-02
 -1.3563693e-02 -4.8062384e-02]
[3.97  3.25  2.529 1.809 1.089 0.368 0.    0.    0.    0.    0.    3.97
 3.25  2.529 1.809 1.089 0.368 0.    0.    0.    0.    0.   ]
0.9980513911979001
[ 2.5349457e+00  2.2012050e+00  1.7706861e+00  1.3582238e+00
  9.2518580e-01  3.9737362e-01 -3.2521293e-02 -8.1371233e-02
 -1.6546607e-02  5.4686517e-02  1.1678660e-01  4.6501594e+00
  3.8348560e+00  3.1367328e+00  2.4241352e+00  1.5522003e+00
  6.8351322e-01  6.8930149e-02 -3.5763085e-03 -1.3394773e-02
 -4.2585984e-02 -3.2487333e-02]
[2.61  2.18  1.751 1.321 0.891 0.462 0.032 0.    0.    0.    0.    4.58
 3.826 3.072 2.318 1.564 0.81  0.056 0.    0.    0.    0.   ]
0.9984525690810293
[

In [12]:
torch.onnx.export(model, torch.zeros(4), './models/est_ground_pressure.onnx')

============= Diagnostic Run torch.onnx.export version 2.0.0+cu117 =============
verbose: False, log level: Level.ERROR
======================= 0 NONE 0 NOTE 0 WARNING 0 ERROR ========================

